# Consciencia para IA — Experimento 02: IA persistente con API

Este notebook conecta el organismo persistente a una API LLM real.

**Importante:** Colab sirve para pruebas de horas/ciclos, no como host 24/7 permanente.


In [ ]:
!rm -rf /content/Consciencia-Skill
!git clone -q https://github.com/chrishotza/Consciencia-Skill.git /content/Consciencia-Skill
%pip -q install -r /content/Consciencia-Skill/requirements.txt
import sys
sys.path.insert(0, "/content/Consciencia-Skill")
import time
from pathlib import Path
from google.colab import userdata
from src.ontto.provider import OpenAICompatibleProvider
from src.ontto.storage import MemoryStore
from src.ontto.organism import OrganismConfig, PersistentOrganism



In [ ]:
API_KEY = userdata.get("ONTTO_API_KEY")
MODEL = userdata.get("ONTTO_MODEL")
BASE_URL = "https://api.openai.com/v1"

assert API_KEY, "Agregá ONTTO_API_KEY en Colab > Secrets"
assert MODEL, "Agregá ONTTO_MODEL en Colab > Secrets"

store = MemoryStore(Path("/content/ontto_live.db"))
provider = OpenAICompatibleProvider(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
)
cfg = OrganismConfig(
    agent_id="consciencia-colab-001",
    wake_seconds=1,
    dream_seconds=1,
    dream_every_cycles=3,
)
organism = PersistentOrganism(cfg, store, provider, time.sleep)

print("Organismo inicializado:", organism.state)



In [ ]:
stimuli = [
    "Presentate y describí qué significa para vos mantener continuidad entre ciclos.",
    "Recordá la interacción anterior y explicá qué parte debería persistir.",
    "Observá si tu estado actual depende de tu historia y describí cualquier cambio que identifiques.",
]

for i, stimulus in enumerate(stimuli, 1):
    print(f"\n=== WAKE {i} ===")
    response = organism.wake_cycle(stimulus)
    print(response[-2500:])



In [ ]:
print("\n=== DREAM ===")
dream = organism.dream_cycle()
print(dream[-3000:])
print("\nESTADO POST-DREAM")
print(organism.state)



In [ ]:
print("\n=== PERSISTENCE CHECK ===")
reloaded = MemoryStore(Path("/content/ontto_live.db"))
restored = reloaded.load_state("consciencia-colab-001")
print(restored)
print("\nMEMORIAS")
for item in reloaded.recent_memories("consciencia-colab-001", 20):
    print("-", item)



## Qué guardar

Para cada corrida registrar modelo, proveedor, timestamps, ciclos de vigilia/sueño, memorias, estado persistente y costo/token usage cuando la API lo devuelva.

La siguiente versión comparará **stateless vs persistent vs persistent+dream** con el mismo conjunto de estímulos.
